# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nandini1313-cloud/flyrank-ml1-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. My rule and its reason codes

### Baseline rule

For the Content Refresh lane, I will prioritize content that is both
stale and still has meaningful search visibility.

The baseline score will combine two signals:

- Staleness: how long it has been since the content was last updated.
- Search visibility: historical Google Search Console impressions.

Higher scores mean higher refresh priority.

### Reason codes

The rule can produce one primary reason code:

- `STALE_HIGH_VISIBILITY` — content is old and has meaningful search visibility.
- `STALE` — content is old but has lower search visibility.
- `HIGH_VISIBILITY` — content has meaningful search visibility but is not very stale.
- `OTHER` — does not strongly match the two conditions above.

### Action labels

- `REFRESH_NOW` — highest priority for content review.
- `REVIEW` — worth reviewing but not the strongest priority.
- `MONITOR` — lower priority for now.

This is a decision-support baseline, not a claim that refreshing a page
will definitely improve its performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB connection ready")

Token loaded: True
DuckDB connection ready


In [6]:
schema = con.sql("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 1
""")

schema

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [7]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for file in files:
    print(file)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [8]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB connection ready")

Token loaded: True
DuckDB connection ready


In [9]:
schema = con.sql("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 1
""")

schema

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [10]:
files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for file in files:
    if "dim_content" in file.lower():
        print(file)

dim_content.parquet


In [11]:
for file in files:
    if "dim_content" in file.lower():
        print(file)

dim_content.parquet


In [12]:
import duckdb
import pandas as pd
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

# Load Hugging Face secret
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Check Colab Secrets.")

# Connect to DuckDB
con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("✅ Hugging Face token loaded")
print("✅ DuckDB connected")

✅ Hugging Face token loaded
✅ DuckDB connected


In [14]:
files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

march_files = [
    f for f in files
    if "fact_content_daily_performance" in f
    and "2026-03" in f
]

for f in march_files:
    print(f)

fact_content_daily_performance/month=2026-03/data_0.parquet


In [18]:
# Find the March 2026 warehouse files

from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for f in files:
    if "fact_content_daily_performance" in f:
        print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [19]:
import duckdb
import pandas as pd
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing. Add it in Colab Secrets.")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

api = HfApi(token=HF_TOKEN)

print("✅ Hugging Face connected")
print("✅ DuckDB connected")

✅ Hugging Face connected
✅ DuckDB connected


In [23]:
march_path = "PASTE_THE_EXACT_2026-03_PATH_HERE"


In [25]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Files containing fact_content_daily_performance:\n")

for f in files:
    if "fact_content_daily_performance" in f:
        print(f)

Files containing fact_content_daily_performance:

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_

In [26]:
march_path = "data/fact_content_daily_performance/month=2026-03/data.parquet"

In [28]:
print(march_path)

data/fact_content_daily_performance/month=2026-03/data.parquet


In [30]:
files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for f in files:
    if "fact_content_daily_performance" in f and "2026-03" in f:
        print(f)

fact_content_daily_performance/month=2026-03/data_0.parquet


In [31]:
march_path = "data/fact_content_daily_performance/month=2026-03/data.parquet"

In [32]:
march_path = march_files[0]

print("Using:")
print(march_path)

Using:
fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
march_files = [
    f for f in files
    if "fact_content_daily_performance" in f
    and "2026-03" in f
]

print("March files found:", len(march_files))

for f in march_files:
    print(f)

In [33]:
df = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        month
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/{march_path}'
    )
    WHERE gsc_data_available IS TRUE
""").df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'month']


,report_date,client_hash_id,content_hash_id,gsc_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,20,0,67,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,1,0,0,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,125,1,616,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,7,0,28,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,11,0,25,2026-03


In [34]:
print(df.shape)

(3611061, 8)


In [35]:
df.head()

,report_date,client_hash_id,content_hash_id,gsc_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,20,0,67,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,1,0,0,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,125,1,616,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,7,0,28,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,11,0,25,2026-03


In [36]:
df["avg_position"] = (
    df["gsc_sum_position"] / df["gsc_impressions"]
)

df["avg_position"] = df["avg_position"].replace(
    [float("inf"), -float("inf")],
    pd.NA
)

df[[
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "avg_position"
]].head(10)

,content_hash_id,gsc_impressions,gsc_clicks,avg_position
0,content_b7e512995f79d5a6,20,0,3.350000
1,content_05597932fe4da067,1,0,0.000000
2,content_7a105f548d9c6916,125,1,4.928000
3,content_905aa32a0230694e,7,0,4.000000
4,content_a3ea9792f793ec72,11,0,2.272727
5,content_36c36abc7650d7af,239,1,7.347280
6,content_a7da352b73b02668,191,0,7.832461
7,content_05434271b257bb68,55,0,3.272727
8,content_d056587ff7faca0c,77,0,5.636364
9,content_bfd1e41c2af250c8,2,0,4.500000


In [37]:
print("Rows:", len(df))
print(df.columns.tolist())
display(df.head())

Rows: 3611061
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'month', 'avg_position']


,report_date,client_hash_id,content_hash_id,gsc_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,month,avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,20,0,67,2026-03,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,1,0,0,2026-03,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,125,1,616,2026-03,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,7,0,28,2026-03,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,11,0,25,2026-03,2.272727


In [38]:
df["avg_position"] = (
    df["gsc_sum_position"] / df["gsc_impressions"].replace(0, pd.NA)
)

display(
    df[
        ["content_hash_id", "gsc_impressions", "gsc_clicks", "avg_position"]
    ].head(10)
)

,content_hash_id,gsc_impressions,gsc_clicks,avg_position
0,content_b7e512995f79d5a6,20,0,3.350000
1,content_05597932fe4da067,1,0,0.000000
2,content_7a105f548d9c6916,125,1,4.928000
3,content_905aa32a0230694e,7,0,4.000000
4,content_a3ea9792f793ec72,11,0,2.272727
5,content_36c36abc7650d7af,239,1,7.347280
6,content_a7da352b73b02668,191,0,7.832461
7,content_05434271b257bb68,55,0,3.272727
8,content_d056587ff7faca0c,77,0,5.636364
9,content_bfd1e41c2af250c8,2,0,4.500000


In [39]:
df["impression_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 100, 1000, 10000, float("inf")],
    labels=["0-100", "101-1K", "1K-10K", "10K+"]
)

impression_table = (
    df.groupby("impression_bucket", observed=False)
      .size()
      .reset_index(name="n")
)

display(impression_table)

,impression_bucket,n
0,0-100,2977578
1,101-1K,601123
2,1K-10K,32232
3,10K+,128


In [40]:
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["Top 3", "4-10", "11-20", "21+"]
)

position_table = (
    df.groupby("position_bucket", observed=False)
      .size()
      .reset_index(name="n")
)

display(position_table)

,position_bucket,n
0,Top 3,564173
1,4-10,1456122
2,11-20,519223
3,21+,908354


In [41]:
baseline = df.copy()

baseline["impression_points"] = pd.cut(
    baseline["gsc_impressions"],
    bins=[-1, 100, 1000, 10000, float("inf")],
    labels=[0, 1, 2, 3]
).astype(int)

baseline["position_points"] = pd.cut(
    baseline["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=[0, 1, 2, 3],
    include_lowest=True
).astype("Int64").fillna(0).astype(int)

baseline["score"] = (
    baseline["impression_points"]
    + baseline["position_points"]
)

In [42]:
baseline["reason_code"] = "LOW_VISIBILITY"

baseline.loc[
    (baseline["gsc_impressions"] > 1000) &
    (baseline["avg_position"] > 10),
    "reason_code"
] = "VISIBILITY_OPPORTUNITY"

baseline.loc[
    (baseline["gsc_impressions"] > 1000) &
    (baseline["avg_position"] <= 10),
    "reason_code"
] = "STRONG_VISIBILITY"

In [43]:
baseline["reason_code"].value_counts()

,count
reason_code,
LOW_VISIBILITY,3578701
STRONG_VISIBILITY,22825
VISIBILITY_OPPORTUNITY,9535


In [44]:
baseline["action"] = "MONITOR"

baseline.loc[
    baseline["score"] >= 5,
    "action"
] = "REFRESH_NOW"

baseline.loc[
    (baseline["score"] >= 3) &
    (baseline["score"] < 5),
    "action"
] = "REVIEW"

In [45]:
baseline = baseline.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

top20 = baseline.head(20)

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

,rank,content_hash_id,score,reason_code,action
0,1,content_e6df0936699f5b8f,6,VISIBILITY_OPPORTUNITY,REFRESH_NOW
1,2,content_66288edeb93b7c4f,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW
2,3,content_66288edeb93b7c4f,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW
3,4,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW
4,5,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW
5,6,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW
6,7,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW
7,8,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW
8,9,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW
9,10,content_74de5f247659e956,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW


In [46]:
from pathlib import Path

Path("work/outputs").mkdir(
    parents=True,
    exist_ok=True
)

baseline[
    [
        "rank",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("✅ Done!")
print("CSV:", "work/outputs/baseline_action_score.csv")
print("Rows:", len(baseline))

✅ Done!
CSV: work/outputs/baseline_action_score.csv
Rows: 3611061


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
].copy()

top20_review["confidence_note"] = (
    "Moderate confidence based on historical search signals."
)

top20_review["what_would_make_it_wrong"] = (
    "The page may not actually need a refresh, or its performance "
    "may depend on factors not included in this baseline."
)

display(top20_review)

,rank,content_hash_id,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_e6df0936699f5b8f,6,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
1,2,content_66288edeb93b7c4f,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
2,3,content_66288edeb93b7c4f,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
3,4,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
4,5,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
5,6,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
6,7,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
7,8,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
8,9,content_e8a52cf3d5988c07,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."
9,10,content_74de5f247659e956,5,VISIBILITY_OPPORTUNITY,REFRESH_NOW,Moderate confidence based on historical search...,"The page may not actually need a refresh, or i..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [48]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Baseline inputs:")
print("1. gsc_impressions")
print("2. avg_position")

print("\nFuture-window inputs: NONE")
print("Label-derived inputs: NONE")
print("Product flags: NONE")
print("Client names/URLs: NONE")

Baseline inputs:
1. gsc_impressions
2. avg_position

Future-window inputs: NONE
Label-derived inputs: NONE
Product flags: NONE
Client names/URLs: NONE


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.